# 02 · Rule-engine topic tagging (deterministic, explainable)
Runs `src/classification/rule_tag_job.py`: the rule engine (`rule_engine.py` + `tagger.py`) applied to every
customer sentence through a vectorized **PySpark `pandas_udf`** (ported from an earlier VOC project of mine).
The rules are **data** (`category_model.json`), so business users change keywords without code changes.
Output `sentence_rule_tags` has one row per (sentence, topic) with the **sentence text** and the words
that triggered the tag.

In [ ]:
import json, os, sys
for k, v in {"catalog": "daria_krasavina", "schema": "voc_valtera",
             "evidence_path": "/Volumes/daria_krasavina/voc_valtera/evidence"}.items():
    dbutils.widgets.text(k, v)
cat, sch, ev = (dbutils.widgets.get(k) for k in ["catalog", "schema", "evidence_path"])
ns = f"{cat}.{sch}"

# The bundle uploads src/ next to this notebook: make the classification modules importable.
CLS = os.path.abspath("../classification")
if CLS not in sys.path:
    sys.path.insert(0, CLS)
import rule_tag_job

summary = rule_tag_job.run(spark, ns, rules_path=os.path.join(CLS, "category_model.json"))

Valtera Motors VOC category model v1.0: 22 topics (15 leaf)
customer sentences in scope: 22745 | tag rows written: 9316 | 15.2s


In [ ]:
freq = spark.sql(f"""SELECT topic_name, parent_topic, sentences_tagged, interactions_tagged
                     FROM {ns}.rule_tag_frequencies WHERE is_leaf ORDER BY sentences_tagged DESC""").toPandas()
examples = spark.sql(f"""SELECT topic_name, sentence_text, matched_terms FROM {ns}.sentence_rule_tags
                         WHERE via = 'direct' ORDER BY rand(3) LIMIT 8""").toPandas()
print(freq.to_string(index=False)); print(); print(examples.to_string(index=False))
os.makedirs(ev, exist_ok=True)
with open(f"{ev}/02_rule_tagging.json", "w") as f:
    json.dump({**summary, "tags_by_topic": freq.to_dict("records"), "example_explanations": examples.to_dict("records")},
              f, indent=2, default=str)

                          topic_name              parent_topic  sentences_tagged  interactions_tagged
          Praise - Positive Feedback         Positive Feedback               609                  609
              Service - Repair Delay        Service & Warranty               590                  590
      Software - Infotainment or App   Software & Connectivity               540                  540
     Dealer - Poor Dealer Experience Dealer & Sales Experience               445                  445
Billing - Payment or Financing Issue       Billing & Financing               429                  429
             Warranty - Claim Denied        Service & Warranty               300                  300
    Safety - Warning Light or Airbag            Vehicle Safety               292                  292
       Software - OTA Update Failure   Software & Connectivity               237                  237
   Charging - Public Charger Failure        Battery & Charging               232  